<a href="https://colab.research.google.com/github/R341CUL/python-together/blob/main/10-projects/04_build_your_own_neural_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project · Build your own neural network

In the ML track you built a net as a one-off script. Here you package it into a
reusable **`NeuralNet` class** you can point at *any* dataset — the same leap
libraries like PyTorch make. Train it, tweak it, break it, make it yours.

Pure NumPy, so it runs in Colab with nothing to install.

In [ ]:
import numpy as np

## A reusable network class

One hidden layer, binary classification. `fit` runs gradient descent; `predict`
returns 0/1; `predict_proba` returns the probability. Read the comments — every
line maps to lesson 7 (`neural_net_from_scratch`).

In [ ]:
class NeuralNet:
    def __init__(self, n_features, hidden=16, lr=0.1, seed=0):
        rng = np.random.default_rng(seed)
        self.w1 = rng.normal(scale=0.4, size=(n_features, hidden))
        self.b1 = np.zeros((1, hidden))
        self.w2 = rng.normal(scale=0.4, size=(hidden, 1))
        self.b2 = np.zeros((1, 1))
        self.lr = lr
        self.history = []

    @staticmethod
    def _relu(z):
        return np.maximum(0, z)

    @staticmethod
    def _sigmoid(z):
        return 1 / (1 + np.exp(-z))

    def _forward(self, X):
        self.z1 = X @ self.w1 + self.b1
        self.h = self._relu(self.z1)
        self.logit = self.h @ self.w2 + self.b2
        return self._sigmoid(self.logit)

    def fit(self, X, y, epochs=500):
        y = y.reshape(-1, 1)
        for epoch in range(epochs):
            pred = self._forward(X)
            loss = np.mean((pred - y) ** 2)
            self.history.append(loss)
            # backprop
            dpred = 2 * (pred - y) / len(X)
            dlogit = dpred * pred * (1 - pred)
            dw2 = self.h.T @ dlogit
            db2 = dlogit.sum(axis=0, keepdims=True)
            dh = dlogit @ self.w2.T
            dz1 = dh * (self.z1 > 0)
            dw1 = X.T @ dz1
            db1 = dz1.sum(axis=0, keepdims=True)
            # step
            self.w1 -= self.lr * dw1; self.b1 -= self.lr * db1
            self.w2 -= self.lr * dw2; self.b2 -= self.lr * db2
        return self

    def predict_proba(self, X):
        return self._forward(X).ravel()

    def predict(self, X):
        return (self.predict_proba(X) > 0.5).astype(int)

## Train it on a real dataset

We use scikit-learn's breast-cancer data (structured, labelled). Scale the
features first — neural nets train badly on raw, differently-sized columns.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0
)
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

net = NeuralNet(n_features=X.shape[1], hidden=16, lr=0.5).fit(X_train_s, y_train, epochs=800)

print("first loss:", round(net.history[0], 4))
print("last  loss:", round(net.history[-1], 4))
print("test accuracy:", round(accuracy_score(y_test, net.predict(X_test_s)), 3))

## 🏋️ Make it yours

Pick at least one:
- Add a **second hidden layer** (another `w`/`b` pair, another forward+backward step).
- Add a `plot_loss()` method using matplotlib and watch the training curve.
- Try `hidden=4` vs `hidden=64` — where does it start overfitting?
- Point it at a different sklearn dataset, or a Kaggle CSV you load with pandas.

In [ ]:
# your experiments here

### Where this goes next
Once you understand this class, PyTorch will feel familiar: `nn.Linear` is your
`w @ x + b`, `loss.backward()` is your backprop block, `optimizer.step()` is your
weight update. You built the real thing — libraries just make it fast.